In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import random
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import scipy.sparse

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score


# ================================================================
# Utility: device / seeding
# ================================================================
def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ================================================================
# Data loading (identical to the GCN sweep notebook)
# ================================================================
def load_clean_dataset(base_dir: str, dataset_name: str) -> pd.DataFrame:
    csv_path = os.path.join(base_dir, "clean", f"{dataset_name}_clean.csv")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Clean CSV not found at: {csv_path}")
    df = pd.read_csv(csv_path)
    expected_cols = {"text", "label", "split"}
    if not expected_cols.issubset(set(df.columns)):
        raise ValueError(f"{csv_path} must contain columns {expected_cols}, but has {set(df.columns)}")
    train_df = df[df["split"] == "train"]
    val_df = df[df["split"] == "val"]
    test_df = df[df["split"] == "test"]
    if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
        raise ValueError(
            f"{dataset_name}: one of the splits is empty. "
            f"train={len(train_df)}, val={len(val_df)}, test={len(test_df)}"
        )
    return df.reset_index(drop=True)


def load_graph_npz(base_dir: str, dataset_name: str, k: int) -> Tuple[np.ndarray, np.ndarray, int]:
    graph_path = os.path.join(base_dir, "graphs", dataset_name, f"knn_k{k}.npz")
    if not os.path.exists(graph_path):
        raise FileNotFoundError(f"Graph file not found at: {graph_path}")
    try:
        sparse_matrix = scipy.sparse.load_npz(graph_path)
        row, col = sparse_matrix.nonzero()
        num_nodes = sparse_matrix.shape[0]
        return row, col, num_nodes
    except Exception as e:
        data = np.load(graph_path)
        if "row" in data and "col" in data:
            row, col = data["row"], data["col"]
        elif "sources" in data and "targets" in data:
            row, col = data["sources"], data["targets"]
        elif "indices" in data and "indptr" in data and "shape" in data:
            sm = scipy.sparse.csr_matrix(
                (np.ones(len(data["indices"])), data["indices"], data["indptr"]), shape=data["shape"]
            )
            row, col = sm.nonzero()
        else:
            raise ValueError(
                f"Graph file {graph_path} does not contain recognizable sparse-graph keys. "
                f"Keys found: {list(data.keys())}. Original error: {e}"
            )
        num_nodes = int(data["num_nodes"]) if "num_nodes" in data else int(max(row.max(), col.max()) + 1)
        return row, col, num_nodes


def encode_texts_for_dataset(df, tokenizer, transformer, device, max_length=128, batch_size=32) -> torch.Tensor:
    texts = df["text"].astype(str).tolist()
    all_embeddings = []
    transformer.eval()
    with torch.no_grad():
        batches = int(np.ceil(len(texts) / batch_size))
        for i in tqdm(range(batches), desc="  Encoding texts"):
            batch_texts = texts[i * batch_size:(i + 1) * batch_size]
            enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = transformer(**enc)
            cls_emb = outputs.last_hidden_state[:, 0, :]
            all_embeddings.append(cls_emb.cpu())
    return torch.cat(all_embeddings, dim=0)


def prepare_all_datasets(base_dir, model_name, max_length, batch_size, k, device, dataset_names):
    print(f"Using device: {device}")
    print(f"Base data dir: {base_dir}")
    print(f"Transformer: {model_name}, k={k}")
    print("Loading transformer and tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    transformer = AutoModel.from_pretrained(model_name).to(device)

    all_data = {}
    for name in dataset_names:
        print(f"\n=== Preparing dataset: {name} ===")
        df = load_clean_dataset(base_dir, name)
        print(f"  Num documents: {len(df)}")

        embeddings = encode_texts_for_dataset(df, tokenizer, transformer, device, max_length, batch_size)
        labels = torch.tensor(df["label"].values, dtype=torch.long)
        split = df["split"].values
        train_mask = torch.tensor(split == "train", dtype=torch.bool)
        val_mask = torch.tensor(split == "val", dtype=torch.bool)
        test_mask = torch.tensor(split == "test", dtype=torch.bool)

        print("  Loading graph...")
        row, col, num_nodes = load_graph_npz(base_dir, name, k)
        if num_nodes != len(df):
            print(f"  [Warning] num_nodes ({num_nodes}) != len(df) ({len(df)}); clipping indices.")
            num_nodes = len(df)
            row = np.clip(row, 0, num_nodes - 1)
            col = np.clip(col, 0, num_nodes - 1)

        edge_index = torch.tensor(np.vstack([row, col]), dtype=torch.long)

        all_data[name] = {
            "df": df, "emb": embeddings, "labels": labels,
            "train_mask": train_mask, "val_mask": val_mask, "test_mask": test_mask,
            "edge_index": edge_index, "num_nodes": num_nodes,
        }
    return all_data


# ================================================================
# NEW: sparse GAT layer (edge-list based, no torch_geometric,
# no dense N x N attention matrix -- required for FakeNewsNet's
# ~23k nodes). Standard additive attention (Velickovic et al. 2018),
# implemented via scatter_reduce for numerically-stable segment softmax.
# ================================================================
class GATLayer(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, dropout: float = 0.0, negative_slope: float = 0.2):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.a_src = nn.Linear(out_dim, 1, bias=False)
        self.a_dst = nn.Linear(out_dim, 1, bias=False)
        self.leakyrelu = nn.LeakyReLU(negative_slope)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor, num_nodes: int) -> torch.Tensor:
        device = x.device
        row, col = edge_index  # row = target node, col = source neighbor
        self_loop = torch.arange(num_nodes, device=device)
        row = torch.cat([row, self_loop], dim=0)
        col = torch.cat([col, self_loop], dim=0)

        Wh = self.W(x)  # N x out_dim

        alpha_src = self.a_src(Wh).squeeze(-1)  # N
        alpha_dst = self.a_dst(Wh).squeeze(-1)  # N
        e = self.leakyrelu(alpha_src[row] + alpha_dst[col])  # E

        # numerically-stable segment softmax over each target node's incoming edges
        seg_max = torch.full((num_nodes,), float("-inf"), device=device)
        seg_max = seg_max.scatter_reduce(0, row, e, reduce="amax", include_self=True)
        e_stable = e - seg_max[row]
        exp_e = torch.exp(e_stable)
        denom = torch.zeros(num_nodes, device=device)
        denom.index_add_(0, row, exp_e)
        denom = denom.clamp(min=1e-12)
        attn = exp_e / denom[row]
        attn = self.dropout(attn)

        out = torch.zeros_like(Wh)
        out.index_add_(0, row, Wh[col] * attn.unsqueeze(-1))
        return out


class HybridGATTransformer(nn.Module):
    def __init__(self, emb_dim, gat_hidden=256, fusion_hidden=256, num_classes=2, dropout=0.3):
        super().__init__()
        self.gat1 = GATLayer(emb_dim, gat_hidden, dropout=dropout)
        self.gat2 = GATLayer(gat_hidden, gat_hidden, dropout=dropout)
        self.fc_fuse = nn.Linear(emb_dim + gat_hidden, fusion_hidden)
        self.fc_out = nn.Linear(fusion_hidden, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, num_nodes):
        g = F.elu(self.gat1(x, edge_index, num_nodes))
        g = F.elu(self.gat2(g, edge_index, num_nodes))
        fused = torch.cat([x, g], dim=-1)
        fused = F.relu(self.fc_fuse(fused))
        fused = self.dropout(fused)
        return self.fc_out(fused)


# ================================================================
# Resampling + class weights (identical to the GCN sweep)
# ================================================================
def resample_train_mask(labels: torch.Tensor, orig_train_mask: torch.Tensor, target_real_ratio: float, seed: int):
    rng = np.random.RandomState(seed)
    train_idx = torch.nonzero(orig_train_mask, as_tuple=True)[0].numpy()
    train_labels = labels[orig_train_mask].numpy()

    real_idx = train_idx[train_labels == 1]
    fake_idx = train_idx[train_labels == 0]
    R, Fk = len(real_idx), len(fake_idx)

    p = target_real_ratio
    new_real_a = R
    new_fake_a = round(R * (1 - p) / p) if p > 0 else Fk
    new_fake_b = Fk
    new_real_b = round(Fk * p / (1 - p)) if p < 1 else R

    if new_fake_a <= Fk:
        new_real, new_fake = new_real_a, new_fake_a
    else:
        new_real, new_fake = new_real_b, new_fake_b

    new_real = max(1, min(new_real, R))
    new_fake = max(1, min(new_fake, Fk))

    chosen_real = rng.choice(real_idx, size=new_real, replace=False)
    chosen_fake = rng.choice(fake_idx, size=new_fake, replace=False)
    chosen = np.concatenate([chosen_real, chosen_fake])

    new_mask = torch.zeros_like(orig_train_mask)
    new_mask[chosen] = True

    actual_ratio = new_real / (new_real + new_fake)
    return new_mask, actual_ratio, new_real, new_fake


def compute_class_weights(y_src: torch.Tensor, mask: torch.Tensor, num_classes: int, device) -> torch.Tensor:
    labels_subset = y_src[mask]
    counts = torch.bincount(labels_subset, minlength=num_classes).float().clamp(min=1.0)
    n = labels_subset.size(0)
    weights = n / (num_classes * counts)
    return weights.to(device)


def lr_lambda(epoch: int, total_epochs: int, warmup_frac: float = 0.1) -> float:
    warmup_epochs = max(1, int(total_epochs * warmup_frac))
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))


# ================================================================
# Single sweep run -- identical structure/diagnostics to the GCN
# version, just using HybridGATTransformer instead of HybridGCNTransformer.
# ================================================================
def train_one_run(
    src_name: str, all_data: Dict[str, Dict], device,
    target_real_ratio: float, seed: int, class_weighted: bool,
    epochs: int = 30, lr: float = 1e-3, weight_decay: float = 1e-4,
    gat_hidden: int = 256, fusion_hidden: int = 256, dropout: float = 0.3,
    grad_clip_norm: float = 1.0, collapse_epoch_start: int = 3, collapse_threshold: float = 0.98,
) -> Dict:
    set_seed(seed)

    data_src = all_data[src_name]
    x_src = data_src["emb"].to(device)
    y_src = data_src["labels"].to(device)
    orig_train_mask = data_src["train_mask"]
    val_mask = data_src["val_mask"].to(device)
    test_mask = data_src["test_mask"].to(device)
    edge_index_src = data_src["edge_index"].to(device)
    num_nodes_src = data_src["num_nodes"]

    resampled_mask, actual_ratio, n_real, n_fake = resample_train_mask(
        data_src["labels"], orig_train_mask, target_real_ratio, seed
    )
    train_mask = resampled_mask.to(device)

    class_weights = compute_class_weights(y_src, train_mask, 2, device) if class_weighted else None

    emb_dim = x_src.size(1)
    model = HybridGATTransformer(
        emb_dim=emb_dim, gat_hidden=gat_hidden, fusion_hidden=fusion_hidden,
        num_classes=2, dropout=dropout,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda ep: lr_lambda(ep, epochs))
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    best_val_f1 = -1.0
    best_state = None
    epoch_first_collapse: Optional[int] = None
    val_f1_curve: List[float] = []
    val_pred_real_frac_curve: List[float] = []
    loss_curve: List[float] = []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(x_src, edge_index_src, num_nodes_src)
        loss = criterion(logits[train_mask], y_src[train_mask])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits_full = model(x_src, edge_index_src, num_nodes_src)
            val_logits = val_logits_full[val_mask]
            val_labels = y_src[val_mask]
            val_pred = val_logits.argmax(dim=-1)
            val_f1 = f1_score(val_labels.cpu().numpy(), val_pred.cpu().numpy(), average="macro")
            pred_real_frac = val_pred.float().mean().item()

        loss_curve.append(loss.item())
        val_f1_curve.append(val_f1)
        val_pred_real_frac_curve.append(pred_real_frac)

        is_collapsed_now = (pred_real_frac >= collapse_threshold) or (pred_real_frac <= (1 - collapse_threshold))
        if is_collapsed_now and epoch >= collapse_epoch_start and epoch_first_collapse is None:
            epoch_first_collapse = epoch

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    model.eval()
    with torch.no_grad():
        logits_final = model(x_src, edge_index_src, num_nodes_src)
        test_logits = logits_final[test_mask]
        test_labels = y_src[test_mask]
        test_pred = test_logits.argmax(dim=-1)
        test_acc = accuracy_score(test_labels.cpu().numpy(), test_pred.cpu().numpy())
        test_f1 = f1_score(test_labels.cpu().numpy(), test_pred.cpu().numpy(), average="macro")
        test_pred_real_frac = test_pred.float().mean().item()

    final_pred_real_frac = val_pred_real_frac_curve[-1]
    collapsed_final = (final_pred_real_frac >= collapse_threshold) or (final_pred_real_frac <= (1 - collapse_threshold))

    return {
        "source": src_name,
        "target_real_ratio": target_real_ratio,
        "actual_train_real_ratio": actual_ratio,
        "n_train_real": n_real,
        "n_train_fake": n_fake,
        "seed": seed,
        "class_weighted": class_weighted,
        "epoch_first_collapse": epoch_first_collapse if epoch_first_collapse is not None else -1,
        "collapsed_final": bool(collapsed_final),
        "best_val_macro_f1": best_val_f1,
        "test_macro_f1": test_f1,
        "test_acc": test_acc,
        "test_pred_real_frac": test_pred_real_frac,
        "val_f1_curve": val_f1_curve,
        "val_pred_real_frac_curve": val_pred_real_frac_curve,
        "loss_curve": loss_curve,
    }


def run_gat_imbalance_sweep(
    base_dir: str,
    sources: List[str] = ["coaid", "fakenewsnet"],
    ratios: List[float] = [0.5, 0.6, 0.7, 0.8, 0.85],
    seeds: List[int] = [0, 1, 2],
    model_name: str = "roberta-base",
    epochs: int = 30,
    max_length: int = 128,
    batch_size: int = 32,
    k: int = 10,
    dropout: float = 0.3,
):
    device = get_device()
    all_data = prepare_all_datasets(
        base_dir=base_dir, model_name=model_name, max_length=max_length,
        batch_size=batch_size, k=k, device=device, dataset_names=sources,
    )

    out_dir = os.path.join(base_dir, "imbalance_sweep_results")
    os.makedirs(out_dir, exist_ok=True)

    summary_rows = []
    curves = {}
    run_counter = 0
    total_runs = len(sources) * len(ratios) * len(seeds) * 2

    print(f"\n=== Starting GAT imbalance sweep: {total_runs} total runs ===")
    for src_name in sources:
        for ratio in ratios:
            for seed in seeds:
                for class_weighted in [False, True]:
                    run_counter += 1
                    run_id = f"{src_name}_ratio{ratio}_seed{seed}_{'weighted' if class_weighted else 'unweighted'}"
                    print(f"\n[{run_counter}/{total_runs}] {run_id}")
                    result = train_one_run(
                        src_name=src_name, all_data=all_data, device=device,
                        target_real_ratio=ratio, seed=seed, class_weighted=class_weighted,
                        epochs=epochs, dropout=dropout,
                    )
                    print(
                        f"  -> best_val_F1={result['best_val_macro_f1']:.4f} "
                        f"test_F1={result['test_macro_f1']:.4f} "
                        f"collapsed_final={result['collapsed_final']} "
                        f"first_collapse_epoch={result['epoch_first_collapse']}"
                    )
                    curves[run_id] = {
                        "val_f1_curve": result.pop("val_f1_curve"),
                        "val_pred_real_frac_curve": result.pop("val_pred_real_frac_curve"),
                        "loss_curve": result.pop("loss_curve"),
                    }
                    summary_rows.append(result)

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(out_dir, "gat_imbalance_sweep_summary.csv")
    summary_df.to_csv(summary_path, index=False)

    curves_path = os.path.join(out_dir, "gat_imbalance_sweep_curves.json")
    with open(curves_path, "w") as f:
        json.dump(curves, f)

    print(f"\n=== GAT sweep complete ===")
    print(f"Saved summary to: {summary_path}")
    print(f"Saved per-epoch curves to: {curves_path}")

    print("\n=== Collapse summary (collapsed_final rate by ratio x class_weighted) ===")
    pivot = summary_df.groupby(["source", "target_real_ratio", "class_weighted"])["collapsed_final"].mean().unstack()
    print(pivot)

    return summary_df


# ================================================================
# Run: same grid as the GCN sweep, for direct comparison
# ================================================================
run_gat_imbalance_sweep(
    base_dir="drive/MyDrive/CSML/data",
    sources=["coaid", "fakenewsnet"],
    ratios=[0.5, 0.6, 0.7, 0.8, 0.85],
    seeds=[0, 1, 2],
    model_name="roberta-base",
    epochs=30,
    max_length=128,
    batch_size=32,
    k=10,
    dropout=0.3,
)
